In [ ]:
!python -m pip install lightning


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 32.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 45.8 MB/s eta 0:00:00


In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 15.8 MB/s eta 0:00:00


In [ ]:
!pip install selfies

In [ ]:
import pandas as pd
import numpy as np
np.random.seed( 42 )
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors, Lipinski ,QED,rdMolDescriptors,RDConfig
from rdkit.Chem import AllChem
from rdkit.Chem import MACCSkeys
from rdkit import DataStructs
import os
import sys
sys.path.append(os.path.join(RDConfig.RDContribDir, 'SA_Score'))
# now you can import sascore!
import sascorer
import selfies as sf
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets
from torchvision import transforms
import lightning.pytorch as pl
from torch.optim.lr_scheduler import StepLR
from transformers import AutoModelForCausalLM, AutoTokenizer
import math

In [ ]:
import re
import torch
from tqdm import tqdm

def remove_spaces_between_brackets(text):
    pattern = r"\]\s+|\s+\["
    return re.sub(pattern, lambda m: m.group().replace(' ', ''), text)

def decoded_strings_to_smiles(decoded_strings):
    """New function - converts already decoded strings to SMILES"""
    reconstructed_molecules = []
    for decoded_str in decoded_strings:
        cleaned_str = remove_spaces_between_brackets(decoded_str)
        try:
            smiles = sf.decoder(cleaned_str)
            reconstructed_molecules.append(smiles)
        except Exception as e:
            print(f"Error decoding '{cleaned_str}': {e}")
            reconstructed_molecules.append(None)  # or some default value
    return reconstructed_molecules

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("cloudrambler/selfiesbert")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/658 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

In [ ]:
class EncoderVar(nn.Module):
    def __init__(self, z_size, base_model):
        super().__init__()

        self.z_size = z_size
        # self.input_shape = input_shape
        self.base_model = base_model
#         output_size = base_model.hidden_size
        # output_size = self.get_output_size()
        output_size =768

        self.lin_mu = nn.Linear(output_size, z_size)
        self.lin_var = nn.Linear(output_size, z_size)


    # def get_output_size(self):
    #     size = self.base_model(torch.zeros(1, *self.input_shape, device= self.device)).size(1)
    #     return size

    def kl_loss(self):
        kl_loss = -0.5*(1 + self.log_var - self.mu**2 - torch.exp(self.log_var))
        return kl_loss

#     def kl_loss(self):
#         prior = torch.distributions.Normal(0, 1)
#         # Define the posterior distribution based on the encoder output
#         posterior = torch.distributions.Normal( self.mu, torch.exp(0.5 * self.log_var))
#         # Compute the KL divergence between the prior and posterior distributions
#         kl_div = torch.distributions.kl_divergence(posterior, prior)#.sum(dim=1)

#         return kl_div

    def forward(self, x,descriptor):
#         import pdb
#         pdb.set_trace()
        # the base model, same as the traditional AE
#         base_out = self.base_model(x,data_lengths.to('cpu'))
        base_out = self.base_model(x,descriptor)#[0]

        # now the encoder produces means (mu) using the lin_mu output layer
        # and log variances (log_var) using the lin_var output layer
        # we compute the standard deviation (std) from the log variance
        self.mu = self.lin_mu(base_out)
        self.log_var = self.lin_var(base_out)
        std = torch.exp(self.log_var/2)

        # that's the internal random input (epsilon)
        eps = torch.randn_like(self.mu)
        # and that's the z vector
        self.z = self.mu + eps * std

        return self.z


# In[48]:




In [ ]:
from transformers import AutoModelForMaskedLM
model = AutoModelForMaskedLM.from_pretrained("cloudrambler/selfiesbert")


model.safetensors:   0%|          | 0.00/345M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

In [ ]:
class BertEncoder(nn.Module):
    def __init__(self,model):
        super().__init__()


        self.embed_tokens= model.base_model.embeddings

#         self.lin_z = nn.Linear(1024+120, 1024)

        self.cnn_Encoder = nn.Conv1d(768 + featuer_len, 768, kernel_size=3, padding=1)

        self.Bert_encoder =model.base_model.encoder


    def forward(self,x,descriptor):
#         import pdb
#         pdb.set_trace()
        # Create attention mask tensor of ones
#         attention_mask = torch.ones(x.size(0),1,x.size(1),x.size(1)).bool().to("cuda")

        attention_mask = x.ne(tokenizer.pad_token_id).unsqueeze(1).unsqueeze(2).to(torch.bool).to(x.device)
        # Create empty layer_head_mask tensor
        layer_head_mask =None

        x= self.embed_tokens(x)

        descriptor = descriptor.unsqueeze(1)
        descriptor = descriptor.repeat(1, x.size(1), 1)

        x = torch.cat((x, descriptor), dim=-1)
#         x = self.lin_z(x)

        # Reshape x for CNN layer
        x = x.permute(0, 2, 1)  # Reshape to (batch_size, features, sequence_length)

        x = self.cnn_Encoder(x)

        # Reshape x back to original shape
        x = x.permute(0, 2, 1)  # Reshape back to (batch_size, sequence_length, features)

        x= self.Bert_encoder(x, attention_mask, layer_head_mask)

        return x[0]

In [ ]:
class AutoregressiveDecoder(nn.Module):
    def __init__(self, vocab_size, z_size, num_descriptors, max_seq_len=256, d_model=768, nhead=12, num_layers=6, dropout=0.1):
        super().__init__()

        self.d_model = d_model
        self.z_size = z_size
        self.num_descriptors = num_descriptors
        self.max_seq_len = max_seq_len
        self.vocab_size = vocab_size

        self.embed_tokens= model.base_model.embeddings

        # Z condition projection
        self.z_proj = nn.Linear(z_size + num_descriptors, d_model)

        # self.cnn_Decoder = nn.Conv1d(z_size
        #                      + num_descriptors
        #                      , d_model, kernel_size=3, padding=1)

        # Transformer decoder layers
        decoder_layer = nn.TransformerDecoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout,
            #layer_norm_eps = 1e-12,
            activation='gelu',
            batch_first=True
        )
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers)

        # Output projection
        self.output_proj = nn.Linear(d_model, vocab_size)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def create_causal_mask(self, seq_len, device):
        """Create a causal (lower triangular) mask for autoregressive generation"""
        mask = torch.triu(torch.ones(seq_len, seq_len, device=device), diagonal=1)
        return mask.bool()

    def forward(self, z, descriptor, target_tokens=None, max_length=None):
        """
        Args:
            z: latent representation [batch_size, seq_len, z_size]
            descriptor: molecular descriptors [batch_size, num_descriptors]
            target_tokens: target sequence for training [batch_size, seq_len] (optional)
            max_length: maximum generation length for inference (optional)
        """

        batch_size = z.size(0)
        device = z.device

        # Prepare condition (z + descriptor)
        descriptor = descriptor.unsqueeze(1).repeat(1, z.size(1), 1)
        condition = torch.cat([z, descriptor], dim=-1).to(self.z_proj.weight.dtype)

        memory = self.z_proj(condition)  # [batch_size, seq_len, d_model]

        # # Apply CNN layer
        # memory = condition.permute(0, 2, 1)  # Reshape to (batch_size, d_model, sequence_length)
        # memory = self.cnn_Decoder(memory.float())

        # memory = memory.permute(0, 2, 1)  # Reshape back to (batch_size, sequence_length, d_model)



        if target_tokens is not None:
            # Training mode - teacher forcing
            seq_len = target_tokens.size(1)

            token_emb = self.embed_tokens(target_tokens)

            # Create causal mask
            tgt_mask = self.create_causal_mask(seq_len, device)

            # Transformer decoder
            output = self.transformer_decoder(
                tgt=token_emb,
                memory=memory,
                tgt_mask=tgt_mask
            )

            # Project to vocabulary
            logits = self.output_proj(output)
            return logits

        else:
            # Inference mode - autoregressive generation
            if max_length is None:
                max_length = self.max_seq_len

            # Start with CLS token (assuming token id 1 is CLS)
            generated = torch.ones(batch_size, tokenizer.cls_token_id, dtype=torch.long, device=device)

            for _ in range(max_length - 1):
                token_emb = self.embed_tokens(generated)

                # Create causal mask
                seq_len = generated.size(1)
                tgt_mask = self.create_causal_mask(seq_len, device)

                # Transformer decoder
                output = self.transformer_decoder(
                    tgt=token_emb,
                    memory=memory,
                    tgt_mask=tgt_mask
                )

                # Get logits for next token
                logits = self.output_proj(output[:, -1:, :])  # Only last position
                next_token = torch.argmax(logits, dim=-1)

                # Append to sequence
                generated = torch.cat([generated, next_token], dim=1)

                # Check if all sequences have generated EOS token (assuming token id 2 is SEP)
                if torch.all(next_token == tokenizer.sep_token_id):  # SEP token
                    break

            return generated
    def generate_autoregressive(self, z,descriptor, max_length, batch_size, device,
                           temperature=1.0, top_k=50, top_p=0.9):
        """
        Autoregressive generation with temperature, top-k, and top-p sampling
        for increased novelty in molecule SELFIES generation.

        Args:
            max_length: maximum generation length
            batch_size: number of sequences to generate
            device: torch device
            temperature: controls randomness (1.0 = original, >1.0 = more random)
            top_k: number of top tokens to consider for sampling
            top_p: cumulative probability threshold for nucleus sampling
        """
        # Start with CLS token
        generated = torch.ones(batch_size, 1, dtype=torch.long, device=device) * tokenizer.cls_token_id
        unfinished = torch.ones(batch_size, dtype=torch.bool, device=device)  # Track which sequences are still generating

        batch_size = z.size(0)
        device = z.device

        # Prepare condition (z + descriptor)
        descriptor = descriptor.unsqueeze(1).repeat(1, z.size(1), 1)
        condition = torch.cat([z, descriptor], dim=-1).to(self.z_proj.weight.dtype)

        memory = self.z_proj(condition)  # [batch_size, seq_len, d_model]

        # Apply CNN layer
        # memory = condition.permute(0, 2, 1)  # Reshape to (batch_size, features, sequence_length)
        # memory = self.cnn_Decoder(memory.float())

        # memory = memory.permute(0, 2, 1)  # Reshape back to (batch_size, sequence_length, features)

        for _ in range(max_length - 1):
            # Only process unfinished sequences
            if not torch.any(unfinished):
                break

            token_emb = self.embed_tokens(generated)

            # Create causal mask
            seq_len = generated.size(1)
            tgt_mask = self.create_causal_mask(seq_len, device)

            # Transformer decoder
            output = self.transformer_decoder(
                tgt=token_emb,
                memory=memory,
                tgt_mask=tgt_mask
            )

            # Get logits for next token
            logits = self.output_proj(output[:, -1:, :])  # Only last position

            # Apply temperature scaling
            if temperature != 1.0:
                logits = logits / temperature

            # Apply top-k and top-p sampling
            next_token = self.sample_next_token(logits, top_k=top_k, top_p=top_p)

            # Only update unfinished sequences
            next_token[~unfinished] = tokenizer.pad_token_id  # Pad finished sequences

            # Append to sequence
            generated = torch.cat([generated, next_token], dim=1)

            # Update unfinished mask (stop when SEP token is generated)
            unfinished = unfinished & (next_token.squeeze(1) != tokenizer.sep_token_id)

        return generated

    def sample_next_token(self, logits, top_k=None, top_p=None):
        """
        Sample next token using temperature, top-k, and top-p sampling.

        Args:
            logits: raw model outputs [batch_size, 1, vocab_size]
            top_k: number of top tokens to consider
            top_p: cumulative probability threshold for nucleus sampling
        """
        logits = logits.squeeze(1)  # [batch_size, vocab_size]

        # Apply top-k filtering
        if top_k is not None and top_k > 0:
            top_k = min(top_k, logits.size(-1))
            indices_to_remove = logits < torch.topk(logits, top_k)[0][..., -1, None]
            logits[indices_to_remove] = -float('Inf')

        # Apply top-p (nucleus) sampling
        if top_p is not None and top_p > 0.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)

            # Remove tokens with cumulative probability above the threshold
            sorted_indices_to_remove = cumulative_probs > top_p
            # Shift the indices to the right to keep the first token above threshold
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0

            for idx in range(logits.size(0)):
                indices_to_remove = sorted_indices[idx][sorted_indices_to_remove[idx]]
                logits[idx][indices_to_remove] = -float('Inf')

        # Sample from the filtered distribution
        probs = F.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)

        return next_token

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.enc = encoder
        self.dec = decoder

    def forward(self, x, descriptor, desirable_descriptors, target_tokens=None):
        # import pdb
        # pdb.set_trace()
        enc_out = self.enc(x, descriptor)

        if target_tokens is not None:
            # Training: use teacher forcing
            dec_out = self.dec(enc_out, desirable_descriptors, target_tokens)
        else:
            # Inference: autoregressive generation
            dec_out = self.dec(enc_out, desirable_descriptors)

        return dec_out

In [ ]:
class Lora_model(pl.LightningModule):
    def __init__(self, model, lr, vocab_size):
        super().__init__()
        self.model = model
        self.lr = lr
        self.vocab_size = vocab_size

        self.train_losses = []
        self.batch_losses = []
        self.similarities = []

        self.log_scale = nn.Parameter(torch.Tensor([0.0]))

    def forward(self, x, descriptor, desirable_descriptors, target_tokens=None):
        return self.model(x, descriptor, desirable_descriptors, target_tokens)

    def configure_optimizers(self):
        optim = torch.optim.Adam(self.parameters(), lr=self.lr)
        scheduler = StepLR(optim, step_size=40, gamma=0.5)
        return [optim], [scheduler]

    def training_step(self, batch, batch_idx):
        tokens, canonical_smiles, descriptor, desirable_descriptors = batch

        # Forward pass with teacher forcing
        logits = self(tokens, descriptor, desirable_descriptors, target_tokens=tokens)

        # Compute reconstruction loss (autoregressive)
        self.recon_loss = autoregressive_loss(logits, tokens, ignore_index=tokenizer.convert_tokens_to_ids('[PAD]'))

        # Compute KL loss
        self.kl_loss = self.model.enc.kl_loss().sum(dim=[1, 2]).sum(dim=0)

        # # Total loss with beta annealing
        # if self.current_epoch > 9:
        #     beta = 0.1
        #     self.total_loss = self.recon_loss + beta * self.kl_loss
        # else:
        #     self.total_loss = self.recon_loss + self.kl_loss

        self.total_loss = self.recon_loss + self.kl_loss

        self.batch_losses.append(np.array([
            self.total_loss.data.item(),
            self.recon_loss.data.item(),
            self.kl_loss.data.item()
        ]))

        return {'loss': self.total_loss}

    def on_train_epoch_start(self):
        self.start = time.time()

    def on_train_epoch_end(self):
        self.train_losses.append(np.array(self.batch_losses).mean(axis=0))
        self.batch_losses = []

        print("Epoch time =", (time.time() - self.start)/60, "minute")
        print(f'Epoch {self.current_epoch} | Loss >> {self.train_losses[-1][0]:.4f}/ \
              {self.train_losses[-1][1]:.4f}/{self.train_losses[-1][2]:.4f}')

        with open("result_autoregressive_vae.txt", "a") as f:
            f.write('Epoch {} | Loss >> {:.4f}/{:.4f}/{:.4f}\n'.format(
                self.current_epoch,
                self.train_losses[-1][0],
                self.train_losses[-1][1],
                self.train_losses[-1][2]
            ))
            f.write('\n' + '_'*80 + '\n')

    def generate(self, descriptor, desirable_descriptors, z=None, max_length=256):
        """Generate molecules autoregressively"""
        self.eval()
        with torch.no_grad():
            if z is None:
                # Sample from prior
                batch_size = descriptor.size(0)
                z = torch.randn(batch_size, max_length, self.model.enc.z_size).to(descriptor.device)

            generated = self.model.dec(z, desirable_descriptors, max_length=max_length)
            return generated

In [ ]:
# Example usage and initialization
vocab_size=model.config.vocab_size
z_size=58

featuer_len = 76    # selected_desc.shape[1]
num_desirable_descriptors =8  # desirable_descriptors.shape[1]

bert_encoder = BertEncoder(model)
encoder_var = EncoderVar(z_size, bert_encoder)

# Initialize autoregressive decoder
decoder = AutoregressiveDecoder(
    vocab_size=vocab_size,
    z_size=z_size,
    num_descriptors=num_desirable_descriptors,
    max_seq_len=150,
    d_model=768,
    nhead=12,
    num_layers=6,
    dropout=0.1)

# Create autoencoder
autoencoder = AutoEncoder(encoder_var, decoder)

In [ ]:
L_model = Lora_model(autoencoder, lr=0.0002, vocab_size=vocab_size)

In [ ]:
device = 'cuda:1' if torch.cuda.is_available() else 'cpu'

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
import torch

# Original error: RuntimeError: Attempting to deserialize object on CUDA device 1 but torch.cuda.device_count() is 1.
# This means the checkpoint was saved for device 'cuda:1' but only 'cuda:0' is available.
# To handle this robustly, we load the checkpoint to CPU first, then move the model to the *actual* available GPU.
checkpoint = torch.load(r"/content/gdrive/MyDrive/Selfies_data/autoregressive_vae_epoch_epoch=29.ckpt", map_location='cpu')
test_model = L_model
test_model.load_state_dict(checkpoint['state_dict'])
test_model.eval()

# Determine the actual device to use
if torch.cuda.is_available():
    actual_device = 'cuda:0' # Since device_count() is 1, cuda:0 is the only option
else:
    actual_device = 'cpu'

test_model.to(actual_device)

Lora_model(
  (model): AutoEncoder(
    (enc): EncoderVar(
      (base_model): BertEncoder(
        (embed_tokens): BertEmbeddings(
          (word_embeddings): Embedding(185, 768, padding_idx=0)
          (position_embeddings): Embedding(512, 768)
          (token_type_embeddings): Embedding(2, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (cnn_Encoder): Conv1d(844, 768, kernel_size=(3,), stride=(1,), padding=(1,))
        (Bert_encoder): BertEncoder(
          (layer): ModuleList(
            (0-11): 12 x BertLayer(
              (attention): BertAttention(
                (self): BertSelfAttention(
                  (query): Linear(in_features=768, out_features=768, bias=True)
                  (key): Linear(in_features=768, out_features=768, bias=True)
                  (value): Linear(in_features=768, out_features=768, bias=True)
                  (dropout): Dropout(p=0.1, inpl

In [ ]:
test_model.model.dec

AutoregressiveDecoder(
  (embed_tokens): BertEmbeddings(
    (word_embeddings): Embedding(185, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (z_proj): Linear(in_features=66, out_features=768, bias=True)
  (transformer_decoder): TransformerDecoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerDecoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (multihead_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bia

In [ ]:
@torch.no_grad()
def generate_molecules_vae(model, tokenizer, descriptors_loader,
                           device="cuda", num_molecules=30000, max_len=150,
                           temperature=1.0, top_k=50, top_p=0.9):
    """
    Generate molecules from a trained VAE model using only the decoder
    with z ~ N(0, I), sampled once globally for all molecules.

    Args:
        model: Trained VAE model
        tokenizer: Tokenizer for encoding/decoding
        descriptors_loader: DataLoader for molecular descriptors
        device: Device to run on
        num_molecules: Number of molecules to generate
        max_len: Maximum sequence length
        temperature: Controls randomness (1.0 = original, >1.0 = more random)
        top_k: Number of top tokens to consider for sampling
        top_p: Cumulative probability threshold for nucleus sampling
    """
    model.eval()
    all_generated = []

    # 🔹 Sample all latent vectors once
    z_all = torch.randn(num_molecules, model.model.dec.z_size, device=device)

    idx = 0
    for batch in tqdm(descriptors_loader, desc="Generating molecules"):
        descriptors = batch[0] # Extract the tensor from the list
        descriptors = descriptors.to(device)
        batch_size = descriptors.size(0)

        # take the corresponding slice of z_all
        if idx + batch_size > num_molecules:
            batch_size = num_molecules - idx
            if batch_size <= 0:
                break
            descriptors = descriptors[:batch_size]

        z_batch = z_all[idx: idx + batch_size]   # [batch, z_size]
        idx += batch_size

        # expand z along sequence length
        z = z_batch.unsqueeze(1).repeat(1, max_len, 1)  # [batch, max_len, z_size]

        # Decode with sampling parameters
        generated_ids = model.model.dec.generate_autoregressive(
            z=z.to(device),
            descriptor=descriptors,
            max_length=max_len,
            batch_size=batch_size,
            device=device,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p
        )

        # Convert ids -> SMILES strings
        for seq in generated_ids:
            seq = seq.tolist()
            if tokenizer.cls_token_id in seq:  # remove BOS
                seq = seq[seq.index(tokenizer.cls_token_id)+1:]

            if tokenizer.sep_token_id in seq:  # cut at EOS
                seq = seq[:seq.index(tokenizer.sep_token_id)]
            decoded = tokenizer.decode(seq, skip_special_tokens=True)
            all_generated.append(decoded)

            if len(all_generated) >= num_molecules:
                return decoded_strings_to_smiles(all_generated)

    return decoded_strings_to_smiles(all_generated)

In [ ]:
test_desirable_descriptors=pd.read_csv(r'/content/for_generation_VEGFR2_desirable_descriptors_30000.csv')


# # In[ ]:
import joblib
loaded_scaler = joblib.load('/content/minmax_scaler_desirable_decoder_descriptors.pkl')

test_desirable_descriptors = loaded_scaler.transform(test_desirable_descriptors[loaded_scaler.feature_names_in_])

from torch.utils.data import DataLoader, TensorDataset

def create_dataloader_from_numpy(array, batch_size=64, shuffle=True):
    data = torch.tensor(array, dtype=torch.float32)
    dataset = TensorDataset(data)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

test_loader = create_dataloader_from_numpy(test_desirable_descriptors, batch_size=512)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.4.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
loaded_scaler.__dict__

{'feature_range': (0, 1),
 'copy': True,
 'clip': False,
 'feature_names_in_': array(['MolLogP', 'MolWt', 'NumHAcceptors', 'NumHDonors',
        'NumRotatableBonds', 'SAscore', 'TPSA', 'qed'], dtype=object),
 'n_features_in_': 8,
 'n_samples_seen_': 1584663,
 'scale_': array([0.09134672, 0.0100018 , 0.07692308, 0.125     , 0.1       ,
        0.16093342, 0.00484262, 1.32067417]),
 'min_': array([ 0.49272423, -2.50062011,  0.        ,  0.        ,  0.        ,
        -0.20428809,  0.        , -0.25255267]),
 'data_min_': array([-5.39400000e+00,  2.50017000e+02,  0.00000000e+00,  0.00000000e+00,
         0.00000000e+00,  1.26939510e+00,  0.00000000e+00,  1.91230113e-01]),
 'data_max_': array([  5.5533    , 349.999     ,  13.        ,   8.        ,
         10.        ,   7.48314486, 206.5       ,   0.94841915]),
 'data_range_': array([ 10.9473    ,  99.982     ,  13.        ,   8.        ,
         10.        ,   6.21374976, 206.5       ,   0.75718903])}

In [ ]:
generated_smiles = generate_molecules_vae(
    model=test_model,
    tokenizer=tokenizer,
    descriptors_loader=test_loader,
    device=actual_device,  # Changed from 'device' to 'actual_device'
    num_molecules=30000,
    max_len=150,
    temperature=1,  # More randomness
    top_k=100,        # Broader sampling
    top_p=1       # More diverse
)
print("generated_smiles",generated_smiles[:10])

Generating molecules:  58%|█████▊    | 34/59 [2:27:01<1:48:10, 259.63s/it]

In [ ]:
# Create a DataFrame
df_gen = pd.DataFrame(generated_smiles, columns=["Smiles"])  # Use first row as header

# Write DataFrame to CSV
df_gen.to_csv(r"smiles_VEGFR2_without_fintuning_autoregressive_desirable_desc_temp=1_epoch_29.csv", index=False)

print("********_______generated smiles saved______********")

print("number of unique generated smiles",len(set(generated_smiles)))


In [ ]:
import pandas as pd
df_VEGFR2 = pd.read_csv("/content/qed_0.5_VEGFR2_desirable_descriptors.csv")["Smiles"]


metrics = get_all_metrics(
    gen=smiles_generated_samples,
    test=list(df_VEGFR2),
    train=list(df_VEGFR2),
    test_scaffolds=list(df_VEGFR2),
    device=device
)